# Spatial aggregation methods

ERA5 and IMD give data on regular grids (e.g. 0.25\u00b0 \u00d7 0.25\u00b0). When we aggregate
to administrative boundaries like districts, smaller districts may contain no
grid points at all, leaving them with missing data.

This notebook compares two aggregation strategies:
- **Point-in-polygon**: average all grid points that fall within each district
- **Nearest centroid**: assign each district the value from the nearest grid point to its centroid

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import varunayan as v
from varunayan.spatial import aggregate_to_polygons, check_grid_coverage, compare_grids

## Download district boundaries and climate data

In [ ]:
import urllib.request, os

districts_url = "https://bharatviz.saketlab.org/India_LGD_districts.geojson"
districts_file = "indian_districts.geojson"
if not os.path.exists(districts_file):
    urllib.request.urlretrieve(districts_url, districts_file)

districts_sf = gpd.read_file(districts_file)
print(f"{len(districts_sf)} districts loaded")

In [ ]:
imd_tmax = v.imd_temperature_geojson(
    request_id="spatial_demo",
    start_year=2023, end_year=2023,
    geojson_file=districts_file,
    var_type="tmax",
)

imd_spatial = (
    imd_tmax.groupby(["longitude", "latitude"])["temperature"]
    .mean().reset_index(name="temp")
)

era5_temp = v.era5ify_geojson(
    request_id="spatial_era5",
    variables="2m_temperature",
    start_date="2023-01-01", end_date="2023-01-31",
    json_file=districts_file,
    frequency="daily", resolution=0.25,
)

era5_spatial = (
    era5_temp.groupby(["longitude", "latitude"])["value"]
    .mean().reset_index(name="temp")
)

print(f"IMD grid points: {len(imd_spatial)}")
print(f"ERA5 grid points: {len(era5_spatial)}")

## Visualizing grid coverage

We use `compare_grids()` to see how grid points align with district boundaries.

In [ ]:
compare_grids(
    {"IMD": imd_spatial},
    polygons=districts_sf,
    title="IMD grid vs district boundaries",
)

In [ ]:
compare_grids(
    {"ERA5": era5_spatial},
    polygons=districts_sf,
    title="ERA5 grid vs district boundaries",
)

In [ ]:
compare_grids(
    {"IMD": imd_spatial, "ERA5": era5_spatial},
    polygons=districts_sf,
    title="IMD vs ERA5 grid points",
)

## Coverage statistics

In [ ]:
imd_cov = check_grid_coverage(imd_spatial, districts_sf, polygon_id_col="district_name")
print("IMD Grid Coverage:")
print(f"  Total districts: {imd_cov['total_polygons']}")
print(f"  Districts with data: {imd_cov['covered_polygons']}")
print(f"  Coverage: {imd_cov['coverage_percent']}%")

print()

era5_cov = check_grid_coverage(era5_spatial, districts_sf, polygon_id_col="district_name")
print("ERA5 Grid Coverage:")
print(f"  Total districts: {era5_cov['total_polygons']}")
print(f"  Districts with data: {era5_cov['covered_polygons']}")
print(f"  Coverage: {era5_cov['coverage_percent']}%")

## Aggregation methods

### Point-in-polygon

Spatial join to find grid points within each polygon, then average.

In [ ]:
result_pip = aggregate_to_polygons(
    imd_spatial, districts_sf, "temp",
    method="point_in_polygon",
    polygon_id_col="district_name",
)
print(f"Districts with data: {result_pip['temp'].notna().sum()} / {len(result_pip)}")

### Nearest centroid

Each polygon gets the value from the grid point nearest its centroid.

In [ ]:
result_nearest = aggregate_to_polygons(
    imd_spatial, districts_sf, "temp",
    method="nearest_centroid",
    polygon_id_col="district_name",
)
print(f"Districts with data: {result_nearest['temp'].notna().sum()} / {len(result_nearest)}")

## Comparing results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

result_pip.plot(
    column="temp", ax=axes[0], legend=True, cmap="plasma",
    missing_kwds={"color": "lightgrey"}, linewidth=0.1, edgecolor="white",
    legend_kwds={"label": "\u00b0C", "shrink": 0.6},
)
axes[0].set_title("Point-in-Polygon", fontweight="bold")
axes[0].axis("off")

result_nearest.plot(
    column="temp", ax=axes[1], legend=True, cmap="plasma",
    missing_kwds={"color": "lightgrey"}, linewidth=0.1, edgecolor="white",
    legend_kwds={"label": "\u00b0C", "shrink": 0.6},
)
axes[1].set_title("Nearest Centroid", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Value comparison

For districts that have data in both methods:

In [ ]:
comparison = result_pip[["district_name", "temp"]].rename(columns={"temp": "pip"}).merge(
    result_nearest[["district_name", "temp"]].rename(columns={"temp": "nearest"}),
    on="district_name",
).dropna()

cor_val = comparison["pip"].corr(comparison["nearest"])

fig, ax = plt.subplots(figsize=(5, 5))
lims = [comparison[["pip", "nearest"]].min().min() - 1,
        comparison[["pip", "nearest"]].max().max() + 1]
ax.plot(lims, lims, "--", color="grey", lw=0.8)
ax.scatter(comparison["pip"], comparison["nearest"], alpha=0.5, s=10, color="#2C3E50")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.set_xlabel("Point-in-polygon (\u00b0C)")
ax.set_ylabel("Nearest centroid (\u00b0C)")
ax.set_title(f"r = {cor_val:.3f} (n = {len(comparison)} districts)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()